# 01 Forward Solution

This notebook creates MEG forward solutions for selected recordings.

Inputs:

- cleaned epochs, cleaned raw, or evoked derivatives for `info`
- coregistration transform `*-trans.fif`
- FreeSurfer source space, e.g. `sub-*-ico5-src.fif`
- BEM solution, e.g. `sub-*-4-1layer-bem-sol.fif`

Output:

- `forward/*_space-ico5_desc-meg-fwd.fif`

Default behavior:

- existing forward solutions are skipped
- missing inputs are reported in status tables
- the batch continues if one recording fails

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import mne
import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.source_modeling import (
    forward_input_overview_to_dataframe,
    forward_results_to_dataframe,
    make_forward_path,
    source_forward_config_to_dataframe,
    source_forward_channel_flags_from_config,
    write_forward_solutions_for_recordings,
)
from meeg_pipeline.workflow import (
    existing_output_policy_for_step,
    iter_recordings,
    selected_recordings_to_dataframe,
    should_overwrite,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


## Interactive backend

In [ ]:
# Optional, useful for later inspection cells.
# Comment this cell if you run the notebook headlessly.

%matplotlib qt

mne.viz.set_browser_backend("qt")
mne.set_log_level("WARNING")

print("MNE browser backend:", mne.viz.get_browser_backend())

## Selection

In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

# Single-file/manual inspection cells at the end of notebooks are disabled by default
# so batch runs over many participants do not stop for plots or ad-hoc file views.
RUN_SINGLE_FILE_INSPECTIONS = False

selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)

## Overwrite policy

Default:

```python
OVERWRITE_STEPS = []
```

Existing forward solutions are skipped. To recompute forward solutions:

```python
OVERWRITE_STEPS = ["forward"]
```

In [ ]:
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "step": "forward",
            "overwrite": should_overwrite("forward", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "forward",
                OVERWRITE_STEPS,
            ),
        }
    ]
)

## Forward parameters

In [ ]:
SOURCE_SPACING = config.source.spacing

# Leave MEG/EEG as None to derive forward-model channel flags from
# channels.analysis in configs/local.yaml. Set explicitly only for manual tests.
MEG = None
EEG = None
FWD_MEG, FWD_EEG = source_forward_channel_flags_from_config(config)

# MNE default-style minimum distance from inner skull surface.
MINDIST = 5.0

# Coregistration transform description used by 1A_anatomy/04_coregistration.ipynb.
# The transform entity scope itself is configured in configs/local.yaml via
# anatomy.coregistration.transform_scope.
TRANS_DESC = "coreg"
TRANS_SCOPE = config.anatomy.coregistration.transform_scope
ALLOW_COMPATIBLE_TRANS_FALLBACK = config.anatomy.coregistration.allow_compatible_fallback

N_JOBS = config.runtime.n_jobs

pd.concat(
    [
        source_forward_config_to_dataframe(config),
        pd.DataFrame(
            [
                {
                    "source_spacing": SOURCE_SPACING,
                    "meg_override": MEG,
                    "eeg_override": EEG,
                    "effective_meg": FWD_MEG,
                    "effective_eeg": FWD_EEG,
                    "mindist": MINDIST,
                    "trans_desc": TRANS_DESC,
                    "trans_scope": TRANS_SCOPE,
                    "allow_compatible_trans_fallback": ALLOW_COMPATIBLE_TRANS_FALLBACK,
                    "n_jobs": N_JOBS,
                }
            ]
        ),
    ],
    axis=1,
)

## Input overview

The preferred `info` input is selected in this order:

1. cleaned epochs
2. cleaned raw
3. first matching evoked file

A row with status `ready` has all required inputs and can be processed.
Rows with status `exists` are skipped by default.
Rows with status beginning with `missing_` need upstream outputs first.

In [ ]:
forward_policy = existing_output_policy_for_step(
    "forward",
    OVERWRITE_STEPS,
)

forward_overview = forward_input_overview_to_dataframe(
    config,
    selected_recordings,
    on_existing=forward_policy,
    spacing=SOURCE_SPACING,
    trans_desc=TRANS_DESC,
)

forward_overview

## Status summary

In [ ]:
if forward_overview.empty:
    pd.DataFrame()
else:
    (
        forward_overview
        .groupby("status", dropna=False)
        .size()
        .reset_index(name="n_recordings")
        .sort_values(["status"])
    )

## Ready jobs

In [ ]:
ready_forward_jobs = forward_overview.query("status == 'ready'").copy()

ready_forward_columns = [
    "subject",
    "session",
    "task",
    "run",
    "fwd_desc",
    "fwd_meg",
    "fwd_eeg",
    "info_input_kind",
    "info_input",
    "trans_status",
    "trans_scope",
    "trans_match",
    "trans_message",
    "trans_path",
    "canonical_trans_path",
    "src_path",
    "bem_path",
    "fwd_path",
]
ready_forward_columns = [
    column for column in ready_forward_columns
    if column in ready_forward_jobs.columns
]

ready_forward_jobs[ready_forward_columns]


## Write forward solutions

This cell processes all selected recordings. Existing outputs are skipped unless
`OVERWRITE_STEPS` contains `"forward"`. Missing inputs and per-recording failures
are returned as status rows rather than aborting the whole batch.

In [ ]:
forward_results = write_forward_solutions_for_recordings(
    config,
    selected_recordings,
    on_existing=forward_policy,
    spacing=SOURCE_SPACING,
    trans_desc=TRANS_DESC,
    meg=MEG,
    eeg=EEG,
    mindist=MINDIST,
    n_jobs=N_JOBS,
    verbose=True,
)

forward_results_df = forward_results_to_dataframe(forward_results)
forward_results_df

## Result summary

In [ ]:
if "forward_results_df" not in globals() or forward_results_df.empty:
    pd.DataFrame()
else:
    (
        forward_results_df
        .groupby("status", dropna=False)
        .size()
        .reset_index(name="n_recordings")
        .sort_values(["status"])
    )

## Inspect one forward solution

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    INSPECT_INDEX = 0

    if forward_results_df.empty:
        inspect_forward_status = pd.DataFrame(
            [{"status": "no_results", "message": "Run the write cell first."}]
        )
    else:
        inspect_row = forward_results_df.iloc[INSPECT_INDEX]
        inspect_path = Path(inspect_row["path"])

        if inspect_path.exists():
            fwd = mne.read_forward_solution(inspect_path, verbose="error")
            inspect_forward_status = pd.DataFrame(
                [
                    {
                        "status": "loaded",
                        "subject": inspect_row["subject"],
                        "session": inspect_row["session"],
                        "task": inspect_row["task"],
                        "run": inspect_row["run"],
                        "n_sources": fwd["nsource"],
                        "n_channels": fwd["nchan"],
                        "source_ori": fwd["source_ori"],
                        "surf_ori": fwd["surf_ori"],
                        "path": str(inspect_path),
                    }
                ]
            )
        else:
            fwd = None
            inspect_forward_status = pd.DataFrame(
                [
                    {
                        "status": "missing_output",
                        "path": str(inspect_path),
                    }
                ]
            )

    inspect_forward_status
else:
    print('Skipped single-file inspection cell 22 in 3_source_modeling/01_forward_solution.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    from IPython.display import display

    print("selected_recordings:")
    display(selected_recordings_to_dataframe(selected_recordings))

    print("forward_overview all:")
    display(forward_overview)

    print("forward_overview status counts:")
    display(
        forward_overview
        .groupby(["task", "status"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["task", "status"])
    )

    print("example-task rows:")
    display(
        forward_overview
        .query("task == 'example'")
        [
            [
                "subject",
                "session",
                "task",
                "run",
                "status",
                "message",
                "fwd_desc",
                "fwd_meg",
                "fwd_eeg",
                "info_input_kind",
                "info_input",
                "trans_status",
                "trans_scope",
                "trans_match",
                "trans_message",
                "trans_path",
                "canonical_trans_path",
                "src_path",
                "bem_path",
                "fwd_path",
            ]
        ]
    )
else:
    print("Skipped detailed forward-selection diagnostics.")
